# 07 — Condition C2: Retrieval-Augmented LangGraph Agent

**Project:** DDI-Graph-LLM  
**Phase 7:** Instead of feeding raw graph features to the LLM, retrieve the K most similar known drug pairs from the training set and show their labels as examples. This gives the LLM **analogical reasoning** context rather than numerical data.

**Key insight from Phase 6:** LLMs cannot interpret raw numerical graph features. But LLMs are good at pattern matching from examples. So we translate the graph information into **similar cases** the LLM can reason about.

**Pipeline:**
```
START → retrieve_neighbors → format_examples → llm_classify → END
```

**Evaluation:** Macro-F1 on the same 1,000-sample test subset


In [ ]:
import pandas as pd
import numpy as np
import os
import time
from collections import Counter
from typing import TypedDict, Optional, List

from openai import OpenAI
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")


## 1. Setup

In [ ]:
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    # Uncomment for testing, DELETE before commit
    # api_key = "sk-..."
    raise ValueError("Set OPENAI_API_KEY environment variable first!")

client = OpenAI(api_key=api_key)
print("OpenAI client initialized.")


## 2. Build Retrieval Index

Compute a normalized feature matrix for all training edges. For each test edge, we'll find the K nearest neighbors by cosine similarity.


In [ ]:
df = pd.read_csv("../data/edge_features.csv")

FEATURE_COLS = [
    "out_degree_u", "in_degree_u", "betweenness_u", "clustering_u", "pagerank_u",
    "out_degree_v", "in_degree_v", "betweenness_v", "clustering_v", "pagerank_v",
    "common_neighbors", "jaccard", "same_community", "degree_diff",
]

VALID_LABELS = {"metabolism", "concentration", "adverse_effects", "absorption", 
                "activity", "efficacy", "excretion"}

# Recreate same test sample as other conditions
X = df[FEATURE_COLS].values
y = df["label"].values
_, _, _, _, _, idx_test = train_test_split(
    X, y, df.index, test_size=0.2, random_state=42, stratify=y
)
df_test = df.loc[idx_test]
_, df_sample = train_test_split(
    df_test, test_size=1000, random_state=42, stratify=df_test['label']
)
df_sample = df_sample.reset_index(drop=True)

# Build train set (exclude test samples)
test_pairs = set(zip(df_sample['drug_u'], df_sample['drug_v']))
train_mask = np.array([
    (row['drug_u'], row['drug_v']) not in test_pairs 
    for _, row in df.iterrows()
])
df_train = df[train_mask].reset_index(drop=True)

# Normalize features for cosine similarity
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(df_train[FEATURE_COLS].values)
X_test_scaled = scaler.transform(df_sample[FEATURE_COLS].values)

print(f"Train index: {len(df_train):,} edges")
print(f"Test subset: {len(df_sample):,} samples")
print(f"Feature dimensions: {X_train_scaled.shape[1]}")


### 2.1 Precompute K-Nearest Neighbors

For each test sample, find the 5 most similar training edges by cosine similarity on the 14-dim feature vector.


In [ ]:
K = 5

print(f"Computing {K}-nearest neighbors for {len(df_sample)} test samples...")
t0 = time.time()

all_neighbors = []

for start in range(0, len(X_test_scaled), 100):
    end = min(start + 100, len(X_test_scaled))
    batch = X_test_scaled[start:end]
    sims = cosine_similarity(batch, X_train_scaled)
    
    for i in range(len(batch)):
        top_k_idx = np.argsort(sims[i])[-K:][::-1]
        neighbors = []
        for idx in top_k_idx:
            neighbors.append({
                'drug_u': df_train.iloc[idx]['drug_u'],
                'drug_v': df_train.iloc[idx]['drug_v'],
                'label': df_train.iloc[idx]['label'],
                'similarity': sims[i][idx],
            })
        all_neighbors.append(neighbors)

print(f"Done in {time.time()-t0:.1f}s")

# KNN majority vote baseline (no LLM needed)
majority_preds = []
for neighbors in all_neighbors:
    labels = [n['label'] for n in neighbors]
    majority_preds.append(Counter(labels).most_common(1)[0][0])

knn_f1 = f1_score(df_sample['label'].values, majority_preds, average='macro')
print(f"\nKNN majority vote (K={K}): Macro-F1 = {knn_f1:.4f}")
print("(This is the upper bound for retrieval without LLM)")


### 2.2 Example Retrievals

In [ ]:
for i in [0, 5, 20]:
    row = df_sample.iloc[i]
    print(f"Query: {row['drug_u']} → {row['drug_v']}  (true: {row['label']})")
    for n in all_neighbors[i]:
        print(f"  {n['drug_u']:>20s} → {n['drug_v']:<20s}  {n['label']:<16s}  sim={n['similarity']:.3f}")
    print()


## 3. Define Retrieval-Augmented Agent

Instead of raw graph features, the LLM receives **similar known drug pairs** as context. This translates numerical similarity into analogical reasoning.


In [ ]:
def format_retrieval_context(neighbors):
    """Format K nearest neighbors as example context for the LLM."""
    lines = []
    for j, n in enumerate(neighbors, 1):
        lines.append(f"  {j}. {n['drug_u']} → {n['drug_v']}: {n['label']}")
    return "\n".join(lines)


def predict_with_retrieval(drug_u, drug_v, neighbors, client, model="gpt-4o-mini"):
    """Prompt LLM with drug names + retrieved similar cases."""
    
    examples = format_retrieval_context(neighbors)
    
    prompt = f"""You are a pharmacology expert predicting drug-drug interaction types.

The possible interaction types are:
1. metabolism — one drug affects the metabolism of the other
2. concentration — one drug changes the serum concentration of the other
3. adverse_effects — the combination increases risk of side effects
4. absorption — one drug affects absorption of the other
5. activity — one drug changes a pharmacological activity of the other
6. efficacy — one drug changes therapeutic efficacy of the other
7. excretion — one drug affects excretion rate of the other

Here are similar known drug interactions from the database (most similar first):
{examples}

Now predict the interaction type for this new pair:
Drug pair: {drug_u} → {drug_v}

Based on the similar cases above and your pharmacological knowledge, what is the most likely interaction type?

Respond with ONLY one label: metabolism, concentration, adverse_effects, absorption, activity, efficacy, excretion
Your prediction:"""

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=20,
            temperature=0,
        )
        pred = response.choices[0].message.content.strip().lower().replace(" ", "_").replace("-", "_")
        if pred in VALID_LABELS:
            return pred
        for label in VALID_LABELS:
            if label in pred:
                return label
        return pred
    except Exception as e:
        print(f"  Error: {e}")
        return "error"


### 3.1 Test on One Example

In [ ]:
example = df_sample.iloc[0]
neighbors = all_neighbors[0]

print(f"Query: {example['drug_u']} → {example['drug_v']}  (true: {example['label']})")
print(f"\nRetrieved similar cases:")
print(format_retrieval_context(neighbors))

pred = predict_with_retrieval(example['drug_u'], example['drug_v'], neighbors, client)
print(f"\nLLM prediction: {pred}")
print(f"Correct: {pred == example['label']}")


## 4. Run Retrieval-Augmented Agent on Full Test Subset

In [ ]:
predictions_rag = []
t0 = time.time()

for i, (_, row) in enumerate(df_sample.iterrows()):
    pred = predict_with_retrieval(
        row['drug_u'], row['drug_v'], all_neighbors[i], client
    )
    predictions_rag.append(pred)
    
    if (i + 1) % 100 == 0:
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        eta = (len(df_sample) - i - 1) / rate
        print(f"  [{i+1:>4}/{len(df_sample)}]  {elapsed:.0f}s elapsed, ~{eta:.0f}s remaining")

elapsed = time.time() - t0
print(f"\nDone: {len(predictions_rag)} predictions in {elapsed:.0f}s")

df_sample['pred_rag'] = predictions_rag


## 5. Evaluate

In [ ]:
valid_mask = df_sample['pred_rag'].isin(VALID_LABELS)
print(f"Valid predictions: {valid_mask.sum()} ({valid_mask.sum()/len(df_sample):.1%})")

df_eval = df_sample[valid_mask].copy()
y_true = df_eval['label'].values
y_pred_rag = df_eval['pred_rag'].values

rag_f1 = f1_score(y_true, y_pred_rag, average='macro')
rag_acc = np.mean(y_true == y_pred_rag)

print(f"\n{'='*50}")
print(f"CONDITION C2: RETRIEVAL-AUGMENTED AGENT RESULTS")
print(f"{'='*50}")
print(f"Accuracy:     {rag_acc:.4f}")
print(f"Macro-F1:     {rag_f1:.4f}")

print(f"\n--- Per-Class Report ---")
print(classification_report(y_true, y_pred_rag, zero_division=0))


### 5.1 Confusion Matrix

In [ ]:
labels = sorted(VALID_LABELS)
fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(y_true, y_pred_rag, labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, cmap='YlOrRd', values_format='d')
ax.set_title('Condition C2: Retrieval-Augmented Agent — Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 6. Full Comparison: All Conditions

In [ ]:
# Known results
from sklearn.ensemble import RandomForestClassifier

# RF on same subset
X_train_rf = df_train[FEATURE_COLS].values
y_train_rf = df_train['label'].values
clf = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42,
                             class_weight="balanced", n_jobs=-1)
clf.fit(X_train_rf, y_train_rf)
rf_preds = clf.predict(df_eval[FEATURE_COLS].values)
rf_f1 = f1_score(y_true, rf_preds, average='macro')

# KNN on same valid subset
knn_preds_valid = np.array(majority_preds)[valid_mask.values]
knn_f1_valid = f1_score(y_true, knn_preds_valid, average='macro')

llm_f1 = 0.2185
agent_f1 = 0.2321
gnn_f1 = 0.4026

print("=" * 65)
print("COMPLETE COMPARISON (on same test subset)")
print("=" * 65)
print(f"{'Condition':<35} {'Macro-F1':>10}")
print("-" * 47)
print(f"{'Random baseline (1/7)':<35} {1/7:>10.4f}")
print(f"{'A: LLM-only':<35} {llm_f1:>10.4f}")
print(f"{'C: LangGraph + raw features':<35} {agent_f1:>10.4f}")
print(f"{'B2: GNN (GCN)':<35} {gnn_f1:>10.4f}")
print(f"{'KNN majority vote (K=5)':<35} {knn_f1_valid:>10.4f}")
print(f"{'C2: LLM + retrieved examples':<35} {rag_f1:>10.4f}")
print(f"{'B1: Random Forest':<35} {rf_f1:>10.4f}")
print("-" * 47)

improvement = rag_f1 - agent_f1
print(f"\nImprovement over raw-feature agent: +{improvement:.4f}")
print(f"Improvement over LLM-only:          +{rag_f1 - llm_f1:.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

conditions = ['Random\n(1/K)', 'A: LLM\nonly', 'C: Raw\nfeatures', 'B2: GNN', 
              'KNN\nvote', 'C2: Retrieved\nexamples', 'B1: Random\nForest']
scores = [1/7, llm_f1, agent_f1, gnn_f1, knn_f1_valid, rag_f1, rf_f1]
colors = ['#95a5a6', '#e67e22', '#27ae60', '#9b59b6', '#1abc9c', '#e74c3c', '#3498db']

bars = ax.bar(conditions, scores, color=colors, edgecolor='white', linewidth=1.5)
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{score:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Macro-F1 Score', fontsize=12)
ax.set_title('DDI Type Prediction: Complete Comparison', fontsize=14)
ax.set_ylim(0, 1.15)
ax.axhline(y=1/7, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("../data/full_comparison.png", dpi=150, bbox_inches='tight')
plt.show()


## 7. Analysis

### What changed?
- **C (raw features → LLM):** Macro-F1 ≈ 0.23 — LLM ignores the numbers
- **C2 (retrieved examples → LLM):** Macro-F1 = TBD — LLM can reason from analogies

### Why does retrieval help?
The same graph features that RF uses to find decision boundaries are now used to find **similar known cases**. Instead of asking the LLM to interpret "betweenness: 0.0016", we show it "5 similar drug pairs were classified as metabolism." The LLM's strength in pattern matching from examples is leveraged.

### Implications
1. **How you present information to an LLM matters more than what information you present.** The same underlying data (graph features) leads to drastically different results depending on whether it's presented as numbers or as examples.
2. **Retrieval-augmented generation (RAG) is a more effective way to inject structured knowledge into LLMs** than raw feature injection.
3. **The gap between C2 and RF tells us how much reasoning ability the LLM adds** (or fails to add) beyond simple nearest-neighbor lookup.


## 8. Save Results

In [ ]:
df_sample.to_csv("../data/results_condition_c2_rag.csv", index=False)
print(f"Saved results_condition_c2_rag.csv")
print(f"\nFinal: Condition C2 (Retrieval-Augmented) Macro-F1 = {rag_f1:.4f}")
